## Batch Parameters and Aggregation
This notebook lets you set parameters, call an R script called consumption_rate.R for one or more CSVs, and aggregate results into a long-format summary where each row is a channel with `uL_mg_hr`, `temp_C`, and metadata.

The main goal is to produce a dataframe for subsequent statistical analyses.

## Configuration

Set paths and default parameters.

In [2]:
import pandas as pd
import subprocess
from pathlib import Path

# Paths
def find_repo_root(start: Path | None = None) -> Path:
    """Locate the repository root when running from a notebook or the project root."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'data').is_dir() and (candidate / 'scripts').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the repository root (expected data/ and scripts/).')

ROOT = find_repo_root()
DATA_DIR = ROOT / 'data' / 'csvs-and-code-jan2026'
OUT_DIR = ROOT / 'data' / 'processed'
R_SCRIPT = ROOT / 'scripts' / 'consumption_rate.R'

# Create output directory if needed
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Mass adjustment options: Control whether to use corrected masses separately for RMR and M
# Corrected mass = mass * (individuals / recovered_individuals) when recovered_individuals is specified
corrected_mass_RMR = True   # Use corrected mass for RMR output
corrected_mass_M = True     # Use corrected mass for M formula

# Default parameters
DEFAULT_SAL = 33
DEFAULT_CONTROL = 'Ch1'
DEFAULT_CHANNELS = ['Ch2', 'Ch3', 'Ch4']
DEFAULT_MASSES = [0.00024, 0.00024, 0.00024]  # grams
DEFAULT_VOL_CONTROL = 0.002  # liters
DEFAULT_VOLUMES = [0.002, 0.002, 0.002]  # liters
DEFAULT_START_HOUR = 1
DEFAULT_END_HOUR = 8
DEFAULT_MICROBIAL_CUTOFF = 5

## Define Trials

Each "run" is a dictionary with trial-specific parameters and meta-data.

### Trial 1 (10 Nov 2025) - Night
- Box2: small vessels (2ml), 1 animal each
- Box3: cylinders (260-268ml), 20 animals each
- Newbox: small vessels (2ml), 1 animal each - using newpyro CSV format

In [3]:
# Define runs here and in subsequent cells
# Each run can override defaults or use them

trial1_runs = [
    {
        'trial': 'trial1',
        'brick': 'box2',
        'csv': DATA_DIR / 'box2-trial1.csv',
        'individuals': [1,1,1],
        'masses': [0.0002263, 0.0002099, 0.0002703],  # measured (1 animal per channel)
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '10Nov2025',
            'environment': 'night'
        }
    },

    {
        'trial': 'trial1',
        'brick': 'box3',
        'csv': DATA_DIR / 'box3-trial1.csv',
        'individuals': [20,20,20],
        #'recovered_individuals': [21,18,15], #technically recovered 21, but setting to 20 assuming we didn't reallly ever add one
        'recovered_individuals': [21,18,15],
        'masses': [0.0076, 0.0065, 0.0055], #measured mass values of recovered animals
        #Subtract 2 ml for each cylinder because we had a stir bar in each
        #'volumes': [0.264, 0.262, 0.260],  # cylinders in ml → L
        'volumes': [0.262, 0.260, 0.258],
        'vol_control': 0.266,  # Ch1 control cylinder
        'metadata': {
            'vessel': 'cylinder',
            'date': '10Nov2025',
            'environment': 'night'
        }
    },
    {
        'trial': 'trial1',
        'brick': 'newbox',
        'csv': DATA_DIR / 'newpyro-trial1.csv',
        'individuals': [1,1,1],
        'masses': [0.0002513, 0.000297, 0.0001811],  # 1 animal per channel
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '10Nov2025',
            'environment': 'night'
        }
    },
]

### Trial 2 (11 Nov 2025) - Night
**Note:** Box2 and Box3 had crashes.
Only Data then is from Newbox: medium vessels (25ml), 5 animals each


In [4]:
trial2_runs = [
    {
        'trial': 'trial2',
        'brick': 'newbox',
        'csv': DATA_DIR / 'newpyro-trial2.csv',  # Using newpyro CSV format
        'individuals': [5,5,5],
        'recovered_individuals': [5,4,5],
        'masses': [0.0011, 0.0009, 0.0011],  # 5 animals × 0.00024 = 0.0012 g
        'volumes': [0.025, 0.025, 0.025],  # 25 ml each
        'vol_control': 0.025,
        'metadata': {
            'vessel': 'medium',
            'date': '11Nov2025',
            'environment': 'night'
        }
    },
]

### Trial 3 (12 Nov 2025) - Night
- Box2: cylinders (260, 262, 264, 268 ml), 20 animals each
- Box3: medium vessels (25ml), 5 animals each
- Newbox: small vessels (2ml), 1 animal each

In [5]:
trial3_runs = [
    {
        'trial': 'trial3',
        'brick': 'box2',
        'csv': DATA_DIR / 'box2-trial3.csv',
        'individuals': [20,20,20],
        'recovered_individuals': [20,20,17],
        #'masses': [0.0049, 0.0047, 0.005058824],  these are corrected masses
        'masses': [0.0049, 0.0047, 0.0043],
        'volumes': [0.262, 0.264, 0.268],  # cylinders Ch2, Ch3, Ch4 in L
        'vol_control': 0.260,  # Ch1 control cylinder

        'metadata': {
            'vessel': 'cylinder',
            'date': '12Nov2025',
            'environment': 'night',
            'notes': 'Ch2 closest to GoPro, Ch4 farthest from GoPro'
        }
    },
    {
        'trial': 'trial3',
        'brick': 'box3',
        'csv': DATA_DIR / 'box3-trial3.csv',
        'individuals': [5,5,5],
        'recovered_individuals': [5,5,4],
        'masses': [0.0011, 0.001, 0.0009],  # 5 animals × 0.00024 = 0.0012 g
        'volumes': [0.025, 0.025, 0.025],  # 25 ml each
        'vol_control': 0.025,
        'metadata': {
            'vessel': 'medium',
            'date': '12Nov2025',
            'environment': 'night'
        }
    },
    {
        'trial': 'trial3',
        'brick': 'newbox',
        'csv': DATA_DIR / 'newpyro-trial3.csv',  # Using newpyro CSV format
        'masses': [0.0002168, 0.0001989, 0.0002552],  # 1 animal per channel
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'microbial_cutoff_hour': 4,  # match the cutoff hour used in R script
        'metadata': {
            'vessel': 'small',
            'date': '12Nov2025',
            'environment': 'night'
        }
    },
]

#### Notes for discussion Trial 3 cylinders do not match XL spreadsheet. I don't understand "mass adjusted" either

- For box2, and box3 it does match
- but what is "mass adjusted" and should that be used instead?

- newbox is cylinders do not match but are also very low (perhaps cylinder order is different).
- In addition, Adriana reports a broken cylinder which is not represented here



### Trial 4 (13 Nov 2025) - Night
- Box2: small vessels (2ml), 1 animal each
- Box3: small vessels (2ml), 1 animal each
- Newbox: cylinders (260, 262, 264, 268 ml), 20 animals each (start_hour=3)

In [6]:
trial4_runs = [
    {
        'trial': 'trial4',
        'brick': 'box2',
        'csv': DATA_DIR / 'box2-trial4.csv',
        'masses': [0.0002187, 0.0003741, 0.0002074],  # 1 animal per channel
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '13Nov2025',
            'environment': 'night'
        }
    },
    {
        'trial': 'trial4',
        'brick': 'box3',
        'csv': DATA_DIR / 'box3-trial4.csv',
        'masses': [0.0002946, 0.0002209, 0.0002518],  # 1 animal per channel
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '13Nov2025',
            'environment': 'night'
        }
    },
    {
        'trial': 'trial4',
        'brick': 'newbox',
        'csv': DATA_DIR / 'newpyro-trial4.csv',
        'individuals': [20, 20],  # Ch2, Ch4 only (Ch3 animals were lost before weighing)
        'channels': ['Ch2', 'Ch4'],
        'recovered_individuals': [17, 20],  # Note Ch3 animals were lost before weighing
        'masses': [0.0045, 0.0049],  # 20 animals per cylinder
        'volumes': [0.262, 0.268],  # cylinders Ch2, Ch4 (from R script: vol02=0.262, vol04=0.268)
        'vol_control': 0.26,  # Ch1 control cylinder (from R script: volmicr=0.26)
        'metadata': {
            'vessel': 'cylinder',
            'date': '13Nov2025',
            'environment': 'night'
        }
    },
]

### Trial 4.5 (14 Nov 2025) - Day (Same animals as Trial 4), using only small vials not cylinders

In [7]:
trial4_5_runs = [
    {
        'trial': 'trial4.5',
        'brick': 'box2',
        'csv': DATA_DIR / 'box2-trial4-light.csv',
        'masses': [0.0002187, 0.0003741, 0.0002074],  # 1 animal per channel
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '14Nov2025',
            'environment': 'day',
            'notes': 'Same animals as trial4'
        }
    },
    {
        'trial': 'trial4.5',
        'brick': 'box3',
        'csv': DATA_DIR / 'box3-trial4-light.csv',
        'channels': ['Ch3', 'Ch4'],          # override defaults (no Ch2 in this file)
        'masses': [0.0002209, 0.0002518],  # 1 animal per channel
        'volumes': [0.002, 0.002],    # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '14Nov2025',
            'environment': 'day',
            'notes': 'Same animals as trial4'
        }
    },
]

### Trial 5 (14 Nov 2025) - Night Environment

In [8]:
trial5_runs = [
    {
        'trial': 'trial5',
        'brick': 'box2',
        'csv': DATA_DIR / 'box2-trial5.csv',
        'individuals': [20,20,20],
        'recovered_individuals': [15, 15, 15],  # Only 2 cylinders, one broke
        'channels': ['Ch2','Ch4'],  # Only 3 channels one cylinder broke
        'masses': [0.0033, 0.0036],  
        'volumes': [0.262, 0.268],  
        'vol_control': 0.260, 
        'metadata': {
            'vessel': 'cylinder',
            'date': '14Nov2025',
            'environment': 'night',
            'notes': 'One cylinder broke, only 3 channels available'
        }
    },
    {
        'trial': 'trial5',
        'brick': 'box3',
        'csv': DATA_DIR / 'box3-trial5.csv',
        'masses': [0.0001893, 0.0003351, 0.0003627],  # 1 animal per channel
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '14Nov2025',
            'environment': 'night'
        }
    },
    {
        'trial': 'trial5',
        'brick': 'newbox',
        'csv': DATA_DIR / 'newpyro-trial5.csv',
        'masses': [0.0002234, 0.0002704, 0.0002277],  # 1 animal per channel
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '14Nov2025',
            'environment': 'night'
        }
    },
]

### Trial 5.5 (15 Nov 2025) - Day Environment (Same animals as Trial 5)

In [9]:
trial5_5_runs = [
    {
        'trial': 'trial5.5',
        'brick': 'newbox',
        'csv': DATA_DIR / 'newpyro-trial5-light.csv',
        'masses': [0.0002234, 0.0002704, 0.0002277],  # 1 animal per channel
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '15Nov2025',
            'environment': 'day',
            'notes': 'Same animals as trial5'
        }
    },
    {
        'trial': 'trial5.5',
        'brick': 'box3',
        'csv': DATA_DIR / 'box3-trial5-light.csv',
        'channels': ['Ch2', 'Ch3', 'Ch4'],  # measurement channels (control from separate CSV)
        'control': 'Ch1',
        'control_csv': DATA_DIR / 'newpyro-trial5-light.csv',
        'masses': [0.0001893, 0.0003351, 0.0003627],  # Same animals as trial5
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '15Nov2025',
            'environment': 'day',
            'notes': 'Control from newpyro-trial5-light.csv'
        }
    },
]

### Trial 6 (16 Nov 2025) - Day 
**Note:** Ch1 serves as control for both box3 and newbox

In [10]:
trial6_runs = [
    {
        'trial': 'trial6',
        'brick': 'box3',
        'csv': DATA_DIR / 'box3-trial6-light.csv',
        'channels': ['Ch2', 'Ch3', 'Ch4'],  # Ch1 missing, get from control_csv
        'control': 'Ch1',
        'control_csv': DATA_DIR / 'newpyro-trial6-light.csv',
        'masses': [0.0002401, 0.0002579, 0.0002079],  # 1 animal per channel
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '16Nov2025',
            'environment': 'day',
            'notes': 'Ch1 control from newpyro-trial6-light.csv'
        }
    },

    {
        'trial': 'trial6',
        'brick': 'newbox',
        'csv': DATA_DIR / 'newpyro-trial6-light.csv',
        'control': 'Ch1',
        'masses': [0.000193, 0.0002411, 0.0002281],  # 1 animal per channel
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '16Nov2025',
            'environment': 'day',
            'notes': 'Ch1 control in same CSV'
        }
    },
]

### Trial 6.5 (16 Nov 2025) - Night Environment (Same animals as Trial 6)

In [11]:
trial6_5_runs = [
    {
        'trial': 'trial6.5',
        'brick': 'box3',
        'csv': DATA_DIR / 'box3-trial6-dark.csv',
        'channels': ['Ch2', 'Ch4'],  # only Ch2 and Ch4 in this CSV (Ch3 missing)
        'control': 'Ch1',
        'control_csv': DATA_DIR / 'newpyro-trial6-dark.csv',  # get control from newbox CSV
        'masses': [0.0002401, 0.0002079],  # only 2 masses for Ch2 and Ch4
        'volumes': [0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '16Nov2025',
            'environment': 'night',
            'notes': 'Ch1 control from newpyro-trial6-dark.csv; Ch3 missing from box3 CSV'
        }
    },
    {
        'trial': 'trial6.5',
        'brick': 'newbox',
        'csv': DATA_DIR / 'newpyro-trial6-dark.csv',
        'control': 'Ch1',  # Ch1 is in the same CSV
        'masses': [0.000193, 0.0002411, 0.0002281],  # 1 animal per channel
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '16Nov2025',
            'environment': 'night',
            'notes': 'Same animals as trial6; Ch1 control in same CSV'
        }
    },
]

### Trial 7 (17 Nov 2025) - Day Environment

In [12]:
trial7_runs = [
    {
        'trial': 'trial7',
        'brick': 'newbox',
        'csv': DATA_DIR / 'newpyro-trial7-light.csv',
        'control': 'Ch1',
        'masses': [0.0002279, 0.0002633, 0.0002362],  # measured masses (N2, N3, N4)
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '17Nov2025',
            'environment': 'day',
            'notes': 'Ch1 control in same CSV'
        }
    },
    {
        'trial': 'trial7',
        'brick': 'box3',
        'csv': DATA_DIR / 'box3-trial7-light.csv',
        'channels': ['Ch2', 'Ch3', 'Ch4'],  # Ch1 missing, get from control_csv
        'control': 'Ch1',
        'control_csv': DATA_DIR / 'newpyro-trial7-light.csv',
        'masses': [0.0002644, 0.0002535, 0.000249],  # measured masses (B3C2, B3C3, B3C4)
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '17Nov2025',
            'environment': 'day',
            'notes': 'Ch1 control from newpyro-trial7-light.csv'
        }
    },
]

### Trial 7.5 (17 Nov 2025) - Night Environment (Same animals as Trial 7)

In [13]:
trial7_5_runs = [
    {
        'trial': 'trial7.5',
        'brick': 'newbox',
        'csv': DATA_DIR / 'newpyro-trial7-dark.csv',
        'control': 'Ch1',
        'masses': [0.0002279, 0.0002633, 0.0002362],  # measured masses (N2, N3, N4) - same animals as trial7
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '17Nov2025',
            'environment': 'night',
            'notes': 'Same animals as trial7; Ch1 control in same CSV'
        }
    },
    {
        'trial': 'trial7.5',
        'brick': 'box3',
        'csv': DATA_DIR / 'box3-trial7-dark.csv',
        'channels': ['Ch2', 'Ch3', 'Ch4'],  # Ch1 missing, get from control_csv
        'control': 'Ch1',
        'control_csv': DATA_DIR / 'newpyro-trial7-dark.csv',
        'masses': [0.0002644, 0.0002535, 0.000249],  # measured masses (B3C2, B3C3, B3C4) - same animals as trial7
        'volumes': [0.002, 0.002, 0.002],  # 2 ml each
        'vol_control': 0.002,
        'metadata': {
            'vessel': 'small',
            'date': '17Nov2025',
            'environment': 'night',
            'notes': 'Same animals as trial7; Ch1 control from newpyro-trial7-dark.csv'
        }
    },
]

### Combine All Trials
This cell aggregates all trial run definitions from above into a single `runs` list.

In [14]:
# Combine all trial runs into a single list
runs = (
    trial1_runs + 
    trial2_runs + 
    trial3_runs + 
    trial4_runs + 
    trial4_5_runs + 
    trial5_runs + 
    trial5_5_runs + 
    trial6_runs + 
    trial6_5_runs + 
    trial7_runs + 
    trial7_5_runs
)

print(f"Total runs: {len(runs)}")
print(f"\nRuns by trial:")
for trial in sorted(set(r['trial'] for r in runs)):
    count = len([r for r in runs if r['trial'] == trial])
    print(f"  {trial}: {count} run(s)")

Total runs: 25

Runs by trial:
  trial1: 3 run(s)
  trial2: 1 run(s)
  trial3: 3 run(s)
  trial4: 3 run(s)
  trial4.5: 2 run(s)
  trial5: 3 run(s)
  trial5.5: 2 run(s)
  trial6: 2 run(s)
  trial6.5: 2 run(s)
  trial7: 2 run(s)
  trial7.5: 2 run(s)


## Functions
Define functions to call the R script for individual runs.

In [15]:
def run_respirometry_analysis(run, defaults=None):
    """
    Call consumption_rate.R for a single run.
    
    Args:
        run: dict with 'csv', 'trial', 'brick', and optional parameters
        defaults: dict with default values for missing parameters
    
    Returns:
        tuple: (success: bool, message: str)
    """
    global corrected_mass_RMR
    
    if defaults is None:
        defaults = {}
    
    csv_path = run['csv']
    if not csv_path.exists():
        return False, f"CSV not found: {csv_path}"
    
    
    
    # Build R script arguments
    args = [
        'Rscript', '--vanilla', str(R_SCRIPT),
        '--csv', str(csv_path),
        '--sal', str(run.get('sal', defaults.get('sal', DEFAULT_SAL))),
        '--control', run.get('control', defaults.get('control', DEFAULT_CONTROL)),
        '--channels', ','.join(run.get('channels', defaults.get('channels', DEFAULT_CHANNELS))),
    ]
    
    # If a separate control CSV is provided, pass it along (support both keys)
    control_csv = run.get('control_csv', run.get('control-csv', None))
    if control_csv is not None:
        args += ['--control_csv', str(control_csv)]
    
    # Calculate masses to pass to R script
    # If corrected_mass_RMR=True and recovered_individuals specified, calculate original inferred masses
    masses_g = run.get('masses', defaults.get('masses', DEFAULT_MASSES))
    individuals = run.get('individuals', None)
    recovered_individuals = run.get('recovered_individuals', None)
    
    if corrected_mass_RMR and individuals is not None and recovered_individuals is not None:
        # Calculate original/inferred masses: corrected_mass * (individuals / recovered_individuals)
        original_masses = [m * (indiv / recov) for m, indiv, recov in zip(masses_g, individuals, recovered_individuals)]
        masses_to_pass = original_masses
    else:
        masses_to_pass = masses_g
    
    args += ['--masses', ','.join(str(m) for m in masses_to_pass)]
    
    args += [
        '--vol_control', str(run.get('vol_control', defaults.get('vol_control', DEFAULT_VOL_CONTROL))),
        '--volumes', ','.join(str(v) for v in run.get('volumes', defaults.get('volumes', DEFAULT_VOLUMES))),
        '--start_hour', str(run.get('start_hour', defaults.get('start_hour', DEFAULT_START_HOUR))),
        '--end_hour', str(run.get('end_hour', defaults.get('end_hour', DEFAULT_END_HOUR))),
        '--microbial_cutoff_hour', str(run.get('microbial_cutoff_hour', defaults.get('microbial_cutoff_hour', DEFAULT_MICROBIAL_CUTOFF))),
        '--out', str(OUT_DIR),
    ]
    
    # Add optional flags if specified
    if run.get('mask_channels', False):
        args += ['--mask_channels', 'true']
    if 'cutoff_inclusive' in run:
        args += ['--cutoff_inclusive', 'true' if run['cutoff_inclusive'] else 'false']
    if run.get('debug', False):
        args += ['--debug', 'true']
    if 'ignore' in run:
        args += ['--ignore', run['ignore']]
    
    # Run R script
    try:
        result = subprocess.run(
            args,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        return True, f"Success: {run['trial']}-{run['brick']}"
    except subprocess.CalledProcessError as e:
        error_msg = f"Failed: {run['trial']}-{run['brick']}\n"
        if e.stderr:
            error_msg += f"Error: {e.stderr}"
        return False, error_msg

## Run Analysis of Single Trial (Mainly for Troubleshooting)

In [16]:
# --- Run and view a single trial (all bricks) ---
# Select all runs for a specific trial
trial_name = 'trial7.5'
trial_runs_to_check = [r for r in runs if r['trial'] == trial_name]

# Mass adjustment parameters (same as in batch aggregation)
ave_mass = 0.000249             # average mass in grams (= 0.249 mg)
scaling_coefficient = -0.25     # scaling exponent for mass adjustment

all_rows = []

for single_run in trial_runs_to_check:
    success, message = run_respirometry_analysis(single_run)
    print(message)
    
    # Load and display the summary for this run
    csv_name = single_run['csv'].name
    summary_path = OUT_DIR / f"{csv_name}_R_summary.csv"
    
    if not summary_path.exists():
        print(f"  Warning: Summary not found for {single_run['brick']}")
        continue
    
    r_df = pd.read_csv(summary_path)
    
    # Get channels and masses from run config
    channels = single_run.get('channels', DEFAULT_CHANNELS)
    masses_g = single_run.get('masses', DEFAULT_MASSES)
    
    # Reshape: gather all *_uL_mg_hr columns into rows
    channel_cols = [col for col in r_df.columns if col.endswith('_uL_mg_hr')]
    
    for col in channel_cols:
        channel = col.replace('_uL_mg_hr', '')
        uL_mg_hr_value = r_df[col].iloc[0]
        
        # Calculate mass-adjusted M (uL_hr) using per-individual channel mass
        try:
            ch_idx = channels.index(channel)
            channel_mass_g = masses_g[ch_idx]
            channel_mass_mg = channel_mass_g * 1000
            
            # Get number of individuals for this channel and calculate corrected mass
            individuals = single_run.get('individuals', None)
            recovered_individuals = single_run.get('recovered_individuals', None)
            
            # corrected_mass_mg (total channel mass) = (what_we_weighed / recovered) * intended_individuals
            corrected_mass_mg = None
            if individuals is not None and recovered_individuals is not None:
                num_individuals = individuals[ch_idx]
                num_recovered = recovered_individuals[ch_idx]
                corrected_mass_mg = (channel_mass_mg / num_recovered) * num_individuals
            
            if individuals is not None:
                num_individuals = individuals[ch_idx]
                # Calculate mass for RMR output (may use corrected or uncorrected)
                if corrected_mass_RMR and recovered_individuals is not None:
                    num_recovered = recovered_individuals[ch_idx]
                    mass_for_rmr_mg = corrected_mass_mg / num_individuals if corrected_mass_mg is not None else channel_mass_mg / num_individuals
                else:
                    mass_for_rmr_mg = channel_mass_mg / num_individuals
                
                # Calculate mass for M formula (may use corrected or uncorrected)
                if corrected_mass_M and recovered_individuals is not None:
                    # Use corrected mass divided by total intended individuals
                    mass_for_m_mg = corrected_mass_mg / num_individuals if corrected_mass_mg is not None else channel_mass_mg / num_individuals
                else:
                    # Use uncorrected mass divided by recovered individuals (actual animals present)
                    if recovered_individuals is not None:
                        num_recovered = recovered_individuals[ch_idx]
                        mass_for_m_mg = channel_mass_mg / num_recovered
                    else:
                        mass_for_m_mg = channel_mass_mg / num_individuals
            else:
                num_individuals = None
                mass_for_rmr_mg = channel_mass_mg
                mass_for_m_mg = channel_mass_mg
            
            ave_mass_mg = ave_mass * 1000
            uL_hr_value = (uL_mg_hr_value / (mass_for_m_mg ** scaling_coefficient)) * (ave_mass_mg ** scaling_coefficient)
        except (ValueError, IndexError):
            num_individuals = None
            mass_for_rmr_mg = None
            mass_for_m_mg = None
            corrected_mass_mg = None
            uL_hr_value = None
        
        row = {
            'brick': single_run.get('brick', None),
            'channel': channel,
            'n': num_individuals,
            'total_mass_mg': channel_mass_mg,
            'corrected_mass_mg': corrected_mass_mg,
            'RMR': uL_mg_hr_value,
            'M': uL_hr_value,
            'temp_C': r_df['temp_C'].iloc[0] if 'temp_C' in r_df.columns else None,
            'vessel': single_run.get('metadata', {}).get('vessel', None),
            'environment': single_run.get('metadata', {}).get('environment', None)
        }
        all_rows.append(row)

# Display combined results for entire trial
if all_rows:
    results_df = pd.DataFrame(all_rows)
    print(f"\n{'='*60}")
    print(f"Results for {trial_name} (all bricks):")
    display(results_df)
    print(f"{'='*60}")

else:    print(f"No results collected for {trial_name}")

Success: trial7.5-newbox
Success: trial7.5-box3

Results for trial7.5 (all bricks):


,brick,channel,n,total_mass_mg,corrected_mass_mg,RMR,M,temp_C,vessel,environment
0,newbox,Ch2,None,0.2279,None,3.967867,3.880997,26.549,small,night
1,newbox,Ch3,None,0.2633,None,2.798130,2.837467,26.549,small,night
2,newbox,Ch4,None,0.2362,None,2.626962,2.592531,26.549,small,night
3,box3,Ch2,None,0.2644,None,1.358387,1.378920,26.417,small,night
4,box3,Ch3,None,0.2535,None,1.360360,1.366465,26.417,small,night
5,box3,Ch4,None,0.2490,None,1.414543,1.414543,26.417,small,night


## Run Batch Data Analyses

In [16]:
# Process all runs
results = []

for run in runs:
    run_id = f"{run['trial']}-{run['brick']}"
    print(f"Processing {run_id}...")
    success, message = run_respirometry_analysis(run)
    results.append({
        'trial': run['trial'],
        'brick': run['brick'],
        'success': success,
        'message': message
    })
    if not success:
        print(f"  ❌ {message}")
    else:
        print(f"  ✓ {message}")

# Summary
results_df = pd.DataFrame(results)
print(f"\n{'='*60}")
print(f"Completed: {results_df['success'].sum()}/{len(results_df)} runs succeeded")
print(f"{'='*60}")
display(results_df)


Processing trial1-box2...
  ✓ Success: trial1-box2
Processing trial1-box3...
  ✓ Success: trial1-box3
Processing trial1-newbox...
  ✓ Success: trial1-newbox
Processing trial2-newbox...
  ✓ Success: trial2-newbox
Processing trial3-box2...
  ✓ Success: trial3-box2
Processing trial3-box3...
  ✓ Success: trial3-box3
Processing trial3-newbox...
  ✓ Success: trial3-newbox
Processing trial4-box2...
  ✓ Success: trial4-box2
Processing trial4-box3...
  ✓ Success: trial4-box3
Processing trial4-newbox...
  ✓ Success: trial4-newbox
Processing trial4.5-box2...
  ✓ Success: trial4.5-box2
Processing trial4.5-box3...
  ✓ Success: trial4.5-box3
Processing trial5-box2...
  ✓ Success: trial5-box2
Processing trial5-box3...
  ✓ Success: trial5-box3
Processing trial5-newbox...
  ✓ Success: trial5-newbox
Processing trial5.5-newbox...
  ✓ Success: trial5.5-newbox
Processing trial5.5-box3...
  ✓ Success: trial5.5-box3
Processing trial6-box3...
  ✓ Success: trial6-box3
Processing trial6-newbox...
  ✓ Success: t

,trial,brick,success,message
0,trial1,box2,True,Success: trial1-box2
1,trial1,box3,True,Success: trial1-box3
2,trial1,newbox,True,Success: trial1-newbox
3,trial2,newbox,True,Success: trial2-newbox
4,trial3,box2,True,Success: trial3-box2
5,trial3,box3,True,Success: trial3-box3
6,trial3,newbox,True,Success: trial3-newbox
7,trial4,box2,True,Success: trial4-box2
8,trial4,box3,True,Success: trial4-box3
9,trial4,newbox,True,Success: trial4-newbox


### Create dataframe for all trials

In [17]:
import pandas as pd

# ============================================================================
# MASS ADJUSTMENT PARAMETERS
# ============================================================================
ave_mass = 0.000249             # average mass in grams (= 0.249 mg)
scaling_coefficient = -0.25     # scaling exponent for mass adjustment
# Formula: uL_hr = (uL_mg_hr / (mass_mg ^ scaling_coefficient)) * (ave_mass_mg ^ scaling_coefficient)
# Note: masses are converted from grams (g) to milligrams (mg) for calculation
# ============================================================================

# Collect all R summary CSVs
summary_rows = []

for run in runs:
    csv_name = run['csv'].name
    summary_path = OUT_DIR / f"{csv_name}_R_summary.csv"
    
    run_id = f"{run['trial']}-{run['brick']}"
    if not summary_path.exists():
        print(f"Warning: Summary not found for {run_id}")
        continue
    
    # Read R summary
    r_df = pd.read_csv(summary_path)
    
    # Get channels from run config
    channels = run.get('channels', DEFAULT_CHANNELS)
    # Get masses from run config (in grams)
    masses_g = run.get('masses', DEFAULT_MASSES)
    
    # Extract per-channel results
    for ch_idx, ch in enumerate(channels):
        col = f'{ch}_uL_mg_hr'
        if col in r_df.columns:
            uL_mg_hr_value = float(r_df[col].iloc[0])
            # Get the mass for this specific channel (in grams) and convert to mg
            channel_mass_g = masses_g[ch_idx]
            channel_mass_mg = channel_mass_g * 1000
            
            # Get number of individuals for this channel and calculate corrected mass
            individuals = run.get('individuals', None)
            recovered_individuals = run.get('recovered_individuals', None)
            
            # corrected_mass_mg (total channel mass) = (what_we_weighed / recovered) * intended_individuals
            corrected_mass_mg = None
            if individuals is not None and recovered_individuals is not None:
                num_individuals = individuals[ch_idx]
                num_recovered = recovered_individuals[ch_idx]
                corrected_mass_mg = (channel_mass_mg / num_recovered) * num_individuals
            
            if individuals is not None:
                num_individuals = individuals[ch_idx]
                # Calculate mass for RMR output (may use corrected or uncorrected)
                if corrected_mass_RMR and recovered_individuals is not None:
                    mass_for_rmr_mg = corrected_mass_mg / num_individuals if corrected_mass_mg is not None else channel_mass_mg / num_individuals
                else:
                    mass_for_rmr_mg = channel_mass_mg / num_individuals
                
                # Calculate mass for M formula (may use corrected or uncorrected)
                if corrected_mass_M and recovered_individuals is not None:
                    # Use corrected mass divided by total intended individuals
                    mass_for_m_mg = corrected_mass_mg / num_individuals if corrected_mass_mg is not None else channel_mass_mg / num_individuals
                else:
                    # Use uncorrected mass divided by recovered individuals (actual animals present)
                    if recovered_individuals is not None:
                        num_recovered = recovered_individuals[ch_idx]
                        mass_for_m_mg = channel_mass_mg / num_recovered
                    else:
                        mass_for_m_mg = channel_mass_mg / num_individuals
            else:
                num_individuals = None
                mass_for_rmr_mg = channel_mass_mg
                mass_for_m_mg = channel_mass_mg
            
            # Calculate mass-adjusted M (uL_hr)
            ave_mass_mg = ave_mass * 1000
            uL_hr_value = (uL_mg_hr_value / (mass_for_m_mg ** scaling_coefficient)) * (ave_mass_mg ** scaling_coefficient)
            
            row = {
                'trial': run['trial'],
                'brick': run['brick'],
                'channel': ch,
                'n': num_individuals,
                'total_mass_mg': channel_mass_mg,
                'corrected_mass_mg': corrected_mass_mg,
                'RMR': uL_mg_hr_value,
                'M': uL_hr_value,
                'temp_C': float(r_df['temp_C'].iloc[0]) if 'temp_C' in r_df.columns else None,
            }
            # Add metadata if present
            if 'metadata' in run:
                row.update(run['metadata'])
            summary_rows.append(row)

# Create aggregated dataframe
if summary_rows:
    agg_df = pd.DataFrame(summary_rows)
    
    # Calculate A: RMR / (mass_per_individual ^ scaling_coefficient)
    # Use corrected_mass_mg if available, otherwise total_mass_mg
    # Divide by n (treat None as 1) to get per-individual mass
    def calculate_A(row):
        # Get mass to use (prefer corrected_mass_mg if available)
        mass_mg = row['corrected_mass_mg'] if pd.notna(row['corrected_mass_mg']) else row['total_mass_mg']
        
        # Get number of individuals (treat None as 1)
        n = row['n'] if pd.notna(row['n']) and row['n'] is not None else 1
        
        # Calculate per-individual mass
        mass_per_individual = mass_mg / n
        
        # Calculate A = RMR / (mass_per_individual ^ scaling_coefficient)
        if pd.notna(row['RMR']) and mass_per_individual > 0:
            return row['RMR'] / (mass_per_individual ** scaling_coefficient)
        else:
            return None
    
    agg_df['A'] = agg_df.apply(calculate_A, axis=1)
    
    # Reorder columns to place A between RMR and M
    cols = list(agg_df.columns)
    if 'A' in cols and 'RMR' in cols and 'M' in cols:
        cols.remove('A')
        rmr_idx = cols.index('RMR')
        cols.insert(rmr_idx + 1, 'A')
        agg_df = agg_df[cols]
    
    # Add filtered flag: False for trials 4 and 4.5, True otherwise
    agg_df['filtered'] = ~agg_df['trial'].isin(['trial4', 'trial4.5'])
    
    # Save aggregated summary
    agg_path = OUT_DIR / 'batch_summary.csv'
    agg_df.to_csv(agg_path, index=False)
    
    print(f"Aggregated summary saved to: {agg_path}")
    print(f"Total rows: {len(agg_df)}")
    # display(agg_df)  # Uncomment to see full dataframe

else:
    print("No results to aggregate.")

Aggregated summary saved to: /Users/oakley/Documents/GitHub/signal_respirometry/data/processed/batch_summary.csv
Total rows: 71


In [18]:
# Display a formatted table (excluding the 'file' and 'notes' columns) for quick review
if 'agg_df' in locals() and not agg_df.empty:
    display_cols = [col for col in agg_df.columns if col not in ('file', 'notes')]
    display(agg_df[display_cols])
else:
    print("No data available for display.")

,trial,brick,channel,n,total_mass_mg,corrected_mass_mg,RMR,A,M,temp_C,vessel,date,environment,filtered
0,trial1,box2,Ch2,1.0,0.2263,NaN,2.708969,1.868423,2.644998,27.948,small,10Nov2025,night,True
1,trial1,box2,Ch3,1.0,0.2099,NaN,3.669335,2.483650,3.515933,27.948,small,10Nov2025,night,True
2,trial1,box2,Ch4,1.0,0.2703,NaN,2.927729,2.111020,2.988427,27.948,small,10Nov2025,night,True
3,trial1,box3,Ch2,20.0,7.6000,7.238095,8.546660,6.628954,9.384154,28.233,cylinder,10Nov2025,night,True
4,trial1,box3,Ch3,20.0,6.5000,7.222222,8.352764,6.475010,9.166227,28.233,cylinder,10Nov2025,night,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,trial7.5,newbox,Ch3,NaN,0.2633,NaN,2.798130,2.004383,2.837467,26.549,small,17Nov2025,night,True
67,trial7.5,newbox,Ch4,NaN,0.2362,NaN,2.626962,1.831360,2.592531,26.549,small,17Nov2025,night,True
68,trial7.5,box3,Ch2,NaN,0.2644,NaN,1.358387,0.974067,1.378920,26.417,small,17Nov2025,night,True
69,trial7.5,box3,Ch3,NaN,0.2535,NaN,1.360360,0.965269,1.366465,26.417,small,17Nov2025,night,True
